# The rise of apprenticeship in French higher education (2020-2026)
### Programming for Data Science, Final project (ESILV, Hugo Alatrista-Salas)

**Team members**
- Maxence COLLIN (team head)
- Clément PHUNG (team member)
- Benjamin ZILBER (team member)

**GitHub repository:** https://github.com/Bahyamin/Apprenticeship-higher-education

**Main dataset:** *Cartographie des formations Parcoursup* (French Ministry of Higher Education, open data, Licence Ouverte 2.0), sessions 2020 to 2026, 157,497 rows.
**Second dataset (extra point):** *InserJeunes, apprenticeship by training centre (CFA)* (French Ministry of Education, DEPP / DARES, Licence Ouverte 2.0).

---

## How to read this notebook

This notebook follows the **KDD process** (Knowledge Discovery in Databases, Fayyad et al., 1996). KDD is the path that goes from raw data to useful knowledge:

1. **Selection**: we choose the data that answers our question.
2. **Pre-processing**: we clean it (missing values, wrong types, inconsistent labels).
3. **Transformation**: we reshape it so that algorithms can read it (transactions, time series, coordinates).
4. **Data mining**: we run the algorithms (grouping queries, pattern mining, forecasting, spatial clustering).
5. **Interpretation**: we explain what the results mean for a real decision.

Every technique is written as a Python function (`def ...`). Each function has a comment block giving its **name, input and output**. All functions are called from a single `main()` function at the end, which also launches the **Dash dashboard**. We explain everything as if the reader had never written a line of Python.

## 1. The dataset in human words

**What is Parcoursup?** Parcoursup is the national website where French high-school students apply to higher education after the *baccalauréat* (the final high-school exam). Every programme that recruits through Parcoursup appears on its public map.

**What does one row describe?** One row is **one programme, offered by one institution, in one year**. Example: *"BTS Management of SMEs, in apprenticeship, offered by school X in Nanterre, in 2025"*.

**What is apprenticeship?** In an apprenticeship programme (*formation en apprentissage*), the student signs a work contract with a company. They alternate between school (a training centre, often called a CFA) and the company, and they are paid. Since a 2018 law, any organisation can open an apprenticeship training centre, and the State has paid hiring aids to companies. Our question is how this policy shows up in the higher-education offer.

**Why is it a multidimensional dataset?** The course defines a multidimensional dataset as having three dimensions, and this one has all three:

| Dimension | Meaning | Columns in our data |
|---|---|---|
| Temporal (when?) | the Parcoursup session | `annee` (2020 to 2026) |
| Spatial (where?) | where the programme takes place | `etab_gps` (latitude, longitude), `region`, `departement`, `commune` |
| Analytical (what?) | the characteristics of the programme | `tf` (type of programme), `app` (apprenticeship or not), `tc` (public or private), `amg` (on-site or remote), `aut` (selection tests), `int` (boarding school) |

**Our research questions**
1. Which programmes and which regions offer the most apprenticeship? (grouping query)
2. Which combinations of characteristics go together with apprenticeship? (frequent pattern mining)
3. How fast has apprenticeship grown, and what can we expect for 2027-2029? (temporal analysis and forecasting)
4. Where are the geographic hubs of apprenticeship, at national level and in the Paris region? (spatial clustering)
5. Bonus: in those training centres, how many apprenticeship contracts are interrupted, and how many apprentices find a job? (second dataset)

## 2. Libraries

A **library** is a toolbox of ready-made functions written by other people. We import each one once, at the top.
- `pandas`: tables (DataFrames) and their manipulation.
- `numpy`: fast maths on arrays of numbers.
- `plotly`: interactive charts and maps (zoom, hover).
- `mlxtend`: frequent itemsets (Apriori) and association rules (lab "Pattern Mining").
- `prophet`: forecasting of time series (lab "Python for analyzing temporal data").
- `scikit-learn`: OPTICS and K-means clustering (lab "Python for spatial information").
- `dash`: the interactive dashboard.

If a library is missing, run `pip install -r requirements.txt` once in a terminal.

In [1]:
# Import libraries
import os                                   # to check whether a file exists on disk
import warnings                             # to hide warnings that are not useful for the reader
import logging                              # to silence the very talkative Prophet logs
import numpy as np                          # numerical computing
import pandas as pd                         # tables (DataFrame) and data manipulation
import plotly.express as px                 # quick interactive charts and maps
import plotly.graph_objects as go           # detailed charts (Sankey diagram, custom layers)
import plotly.io as pio                     # rendering options for plotly
from mlxtend.preprocessing import TransactionEncoder               # list of items -> True/False table
from mlxtend.frequent_patterns import apriori, association_rules  # pattern mining algorithms
from sklearn.cluster import OPTICS, KMeans                        # spatial clustering algorithms
from sklearn.metrics import silhouette_score                      # quality score of a clustering
from dash import Dash, dcc, html                                  # dashboard building blocks

warnings.filterwarnings("ignore")                       # keep the notebook readable
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)  # Prophet uses cmdstanpy internally
logging.getLogger("prophet").setLevel(logging.ERROR)
pio.renderers.default = "plotly_mimetype+notebook_connected"  # charts render in Jupyter and in exported files
pd.set_option("display.max_colwidth", 90)               # show longer text in tables
pd.set_option("display.width", 200)

try:                                        # Prophet can be hard to install on some computers,
    from prophet import Prophet             # so we keep a simple fallback (linear trend) if it fails
    PROPHET_OK = True
    logging.getLogger("cmdstanpy").disabled = True   # hide the "Chain [1] start processing" messages
except ImportError:
    PROPHET_OK = False
print("Prophet available:", PROPHET_OK)

Prophet available: True


In [2]:
# Project settings (change them here, nowhere else)
TEAM = ["Maxence COLLIN (team head)",
        "Clément PHUNG (team member)",
        "Benjamin ZILBER (team member)"]
GITHUB_URL = "https://github.com/Bahyamin/Apprenticeship-higher-education"

# Local files (stored in the GitHub repository, compressed) and official URLs used if the files are missing
MAIN_FILE = "data/fr-esr-cartographie_formations_parcoursup.csv.gz"
MAIN_URL = ("https://data.enseignementsup-recherche.gouv.fr/api/explore/v2.1/catalog/datasets/"
            "fr-esr-cartographie_formations_parcoursup/exports/csv?delimiter=%3B")
EXT_FILE = "data/fr-en-inserjeunes-cfa.csv"
EXT_URL = ("https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/"
           "fr-en-inserjeunes-cfa/exports/csv?delimiter=%3B")

FOCUS_YEAR = 2026          # most recent Parcoursup session, used for "today" snapshots
FORECAST_YEARS = 3         # we forecast 2027, 2028 and 2029
RUN_DASHBOARD = True       # set to False if you only want the analysis
DASHBOARD_PORT = 8050      # the dashboard opens at http://127.0.0.1:8050

## 3. Step 1 of KDD: selection and first look at the data

### 3.1 Loading the data
A CSV file is a text file where each line is a row and values are separated by a character (here `;`). The function below reads it into a **DataFrame**, the pandas table structure. A DataFrame is the right structure here because our data is a table: rows are programmes, columns are attributes, and pandas gives us fast filtering, grouping and joining (course lab "Python Fundamentals").

In [3]:
# Function: load_data
# Goal: read a CSV file into a pandas DataFrame; if the local file does not exist, download it from the official URL
# Input: file_path (str) path to the local CSV or CSV.GZ file
#        url (str or None) official download link used only if the local file is missing
#        sep (str) column separator used in the file
# Output: a pandas DataFrame with the raw data
def load_data(file_path, url=None, sep=";"):
    if os.path.exists(file_path):                       # first choice: the copy stored in the repository
        source = file_path
    elif url is not None:                               # second choice: the official open-data portal
        source = url
    else:
        raise FileNotFoundError(f"{file_path} not found and no URL given")
    df = pd.read_csv(source, sep=sep, low_memory=False) # pandas detects .gz compression automatically
    df.columns = [c.strip().lstrip("﻿") for c in df.columns]  # remove invisible characters in headers
    print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {source}")
    return df

### 3.2 Summarising the data
Before any analysis we need to *know* the data. The function below builds a **data dictionary**, one line per column, with:
- the **data type** (`int64` = whole number, `float64` = decimal number, `object`/`str` = text),
- the **number and percentage of missing values** (empty cells),
- the **number of distinct values**,
- the **range of values**: minimum and maximum for numbers, a few examples for text.

In [4]:
# Function: summarize_data
# Goal: describe a DataFrame column by column (type, missing values, distinct values, range)
# Input: df (DataFrame) any table
# Output: a DataFrame called "data dictionary" with one row per column
def summarize_data(df):
    rows = []
    for col in df.columns:                                   # we look at the columns one by one
        s = df[col]
        if pd.api.types.is_numeric_dtype(s):                 # numbers: range = [min, max]
            value_range = f"[{s.min()} , {s.max()}]"
        else:                                                # text: show the 3 most frequent values
            top = s.dropna().astype(str).value_counts().head(3).index.tolist()
            value_range = " | ".join(v[:40] for v in top)
        rows.append({"column": col,
                     "type": str(s.dtype),
                     "missing": int(s.isna().sum()),
                     "missing_%": round(100 * s.isna().mean(), 1),
                     "distinct_values": int(s.nunique()),
                     "range_or_examples": value_range})
    return pd.DataFrame(rows)

## 4. Steps 2 and 3 of KDD: pre-processing and transformation

Raw data is rarely ready. Here are the decisions we take, and **why** (the course asks us to justify every removed feature):

| Decision | Reason |
|---|---|
| Remove `fiche`, `dataviz`, `etab_url` | web links, useful to a human but they carry no information for analysis |
| Remove `nmc`, `gti`, `rnd`, `code_formation`, `etablissement_id_paysage`, `composante_id_paysage` | internal technical identifiers; `gta` is kept because it identifies a programme |
| Split `etab_gps` ("48.85, 2.35") into two numeric columns `lat` and `lon` | clustering algorithms need numbers, not text |
| Keep only the **first** value of `tf` when it lists several types (e.g. "Etudes de santé,Licence") | a programme gets one main type, so counts are not duplicated |
| Merge "DUT" into "BUT" | the DUT was replaced by the BUT in 2021 (national reform), same programmes under a new name |
| Merge "Formation aux métiers du sport" and "Formations aux métiers du sport" | same category spelled two ways in different years |
| Translate types into short English labels | readability for an international audience |
| `app` empty means "not in apprenticeship" | the column is filled only for apprenticeship programmes, so an empty cell is a real "no", not missing data |
| Group the 6 values of `tc` into 3 sectors | "Privés", "Privés hors contrat" and "Privés enseignement supérieur" are all private without a teaching contract with the State |
| Create flags: `remote` (partly or fully remote teaching), `selective` (selection tests), `boarding`, `adapted_schedule` | turn long texts into simple yes/no features that pattern mining can use |
| Flag `metropolitan` (mainland France and Corsica) | for maps and clustering we keep mainland France; overseas territories and campuses abroad (298 rows without region, e.g. Singapore, Barcelona) are far away and would distort distances |
| Use the most recent name of each institution (`etab_uai` code) | names change over the years (rebranding), the UAI code does not |

**UAI** is the official national identifier of a French school (like a social-security number for schools). We use it later to join our two datasets.

In [5]:
# Short English labels for the types of programmes (column "tf")
TYPE_LABELS = {
    "BTS - BTSA - BTSM": "BTS (2-year technician degree)",
    "Licence": "Licence (university bachelor)",
    "Licence sélective": "Selective licence",
    "Etudes de santé": "Health studies",
    "CPGE": "CPGE (preparatory class)",
    "BUT": "BUT (3-year technology bachelor)",
    "DUT": "BUT (3-year technology bachelor)",
    "Formations d'art": "Art and design",
    "Formations diplômantes du travail social": "Social work diplomas",
    "Formations des écoles d'ingénieurs": "Engineering schools",
    "Formations professionnelles": "Professional certifications",
    "Certificats de spécialisation et FCIL": "Specialisation certificates",
    "Mentions complémentaires": "Complementary vocational mentions",
    "Formations des écoles de commerce et de management": "Business schools",
    "Formations aux métiers du sport": "Sport professions",
    "Formation aux métiers du sport": "Sport professions",
    "DCG": "DCG (accounting)",
    "Formations aux métiers de l'hôtellerie restauration": "Hospitality",
    "Formations d'architecture": "Architecture",
    "Diplômes d'université ou d'établissement": "University diplomas",
    "DEUST": "DEUST (2-year university diploma)",
    "C.M.I - Cursus Master en Ingénierie": "Master in engineering (CMI)",
    "Formations préparatoires à l'enseignement supérieur": "Preparatory years",
    "Sciences Po - Instituts d'études politiques": "Sciences Po",
    "I.A.E - Instituts d'administration des entreprises": "IAE (management)",
    "Licence professionnelle": "Professional licence",
}
# The 6 legal statuses of "tc" grouped into 3 sectors
SECTOR_LABELS = {
    "Publics": "Public",
    "Privés sous contrat": "Private, State contract",
    "Privés en contrat avec l’Etat / EESC": "Private, State contract",
    "Privés": "Private, no State contract",
    "Privés hors contrat": "Private, no State contract",
    "Privés enseignement supérieur": "Private, no State contract",
}
# Columns removed, with the reason (kept in a table so that the decision is documented)
DROPPED_COLUMNS = {
    "fiche": "web link", "dataviz": "web link", "etab_url": "web link",
    "nmc": "technical code", "gti": "technical code", "rnd": "technical code",
    "code_formation": "technical code", "etablissement_id_paysage": "technical code",
    "composante_id_paysage": "technical code",
}

In [6]:
# Function: clean_parcoursup
# Goal: apply all pre-processing decisions listed in the table above
# Input: raw (DataFrame) the raw Parcoursup table returned by load_data
# Output: a clean DataFrame with new readable columns (lat, lon, type, sector, is_app, flags...)
def clean_parcoursup(raw):
    df = raw.drop(columns=[c for c in DROPPED_COLUMNS if c in raw.columns]).copy()  # remove useless columns
    df = df.drop_duplicates()                                    # a programme must appear once per year
    gps = df["etab_gps"].str.split(",", expand=True)             # "48.85, 2.35" -> two pieces of text
    df["lat"] = pd.to_numeric(gps[0], errors="coerce")           # text -> number (latitude)
    df["lon"] = pd.to_numeric(gps[1], errors="coerce")           # text -> number (longitude)
    first_type = df["tf"].str.split(",").str[0].str.strip()     # keep the main type only
    df["type"] = first_type.map(TYPE_LABELS).fillna(first_type)  # English label (raw label if unknown)
    df["sector"] = df["tc"].map(SECTOR_LABELS).fillna("Other")   # 6 statuses -> 3 sectors
    df["is_app"] = df["app"].notna()                             # True = programme in apprenticeship
    amg = df["amg"].fillna("")                                   # teaching arrangements (text)
    df["remote"] = amg.str.contains("distance")                  # partly or fully remote teaching
    df["adapted_schedule"] = amg.str.contains("aménagements")    # schedules adapted to athletes, artists...
    df["selective"] = df["aut"].fillna("").str.contains("épreuves")  # selection tests to get in
    df["boarding"] = df["int"].notna()                           # boarding school available
    df["metropolitan"] = df["lat"].between(41, 51.5) & df["lon"].between(-5.5, 10)  # mainland France box
    latest_name = (df.sort_values("annee").groupby("etab_uai")["etab_nom"].last())   # most recent name per UAI
    df["institution"] = df["etab_uai"].map(latest_name)
    df["annee"] = df["annee"].astype(int)                        # the year is a whole number
    print(f"Clean table: {len(df):,} rows, {df['etab_uai'].nunique():,} institutions, "
          f"{df['is_app'].sum():,} apprenticeship rows")
    return df

## 5. Step 4 of KDD: data mining, four queries

### 5.1 Query 1 (grouping): where is apprenticeship offered?

**Technique.** A *grouping query* splits the rows into groups (by type of programme, by region...) and computes one number per group, like `GROUP BY` in SQL (labs "Analyzing simple and multi-files datasets" and "Pandas").

**Indicators** (computed for the most recent session, 2026):

| Indicator | How it is calculated | How to read it |
|---|---|---|
| Number of programmes | count of rows in the group | size of the offer |
| Number of apprenticeship programmes | count of rows where `is_app` is True | size of the apprenticeship offer |
| **Apprenticeship share (%)** | apprenticeship programmes / all programmes x 100 | 50% means one programme out of two can be followed as an apprentice |
| Institutions offering apprenticeship | number of distinct UAI codes with at least one apprenticeship programme | how many schools or training centres are involved |
| Top institutions | institutions ranked by their number of apprenticeship programmes | the largest apprenticeship providers |

In [7]:
# Function: query_grouping
# Goal: Query 1, count programmes and apprenticeship programmes by type, by region and by institution
# Input: df (DataFrame) clean data; year (int) Parcoursup session to analyse; top_n (int) size of the ranking
# Output: a dict of three DataFrames: "by_type", "by_region", "top_institutions"
def query_grouping(df, year, top_n=15):
    d = df[df["annee"] == year]                                  # snapshot of one session
    def summary(group_col):                                      # small helper used twice
        g = d.groupby(group_col).agg(programmes=("gta", "count"),      # all programmes
                                     apprenticeship=("is_app", "sum"), # apprenticeship programmes
                                     institutions_with_app=("etab_uai", # schools with >= 1 app. programme
                                         lambda u: u[d.loc[u.index, "is_app"]].nunique()))
        g["app_share_%"] = (100 * g["apprenticeship"] / g["programmes"]).round(1)
        return g.sort_values("apprenticeship", ascending=False).reset_index()
    top = (d[d["is_app"]].groupby(["institution", "commune", "sector"])   # rank institutions
             .size().rename("apprenticeship_programmes")
             .sort_values(ascending=False).head(top_n).reset_index())
    return {"by_type": summary("type"), "by_region": summary("region"), "top_institutions": top}

In [8]:
# Function: plot_grouping
# Goal: draw the results of Query 1 (two bar charts)
# Input: q1 (dict) output of query_grouping; year (int) for the titles
# Output: a dict of two plotly figures: "q1_types" and "q1_regions"
def plot_grouping(q1, year):
    t = q1["by_type"].head(10).sort_values("apprenticeship")     # 10 biggest types, smallest at the bottom
    fig_t = px.bar(t, x="apprenticeship", y="type", orientation="h",
                   text="app_share_%", color="app_share_%", color_continuous_scale="Blues",
                   labels={"apprenticeship": "Apprenticeship programmes", "type": "",
                           "app_share_%": "Share in app. (%)"},
                   title=f"Apprenticeship programmes by type ({year}); label = share of the type offered in apprenticeship (%)")
    fig_t.update_traces(texttemplate="%{text}%", textposition="outside")
    r = q1["by_region"].sort_values("apprenticeship")
    fig_r = px.bar(r, x="apprenticeship", y="region", orientation="h",
                   text="app_share_%", color="app_share_%", color_continuous_scale="Blues",
                   labels={"apprenticeship": "Apprenticeship programmes", "region": "",
                           "app_share_%": "Share in app. (%)"},
                   title=f"Apprenticeship programmes by region ({year})")
    fig_r.update_traces(texttemplate="%{text}%", textposition="outside")
    for f in (fig_t, fig_r):
        f.update_layout(height=520, margin=dict(l=10, r=40, t=60, b=10))
    return {"q1_types": fig_t, "q1_regions": fig_r}

### 5.2 Query 2 (frequent pattern mining): what goes together with apprenticeship?

**Technique (lab "Pattern Mining").** We see each programme as a **transaction**, the same way a supermarket sees a shopping basket. The "products" in the basket are its characteristics, called **items**: `Type: BTS`, `Sector: Public`, `Apprenticeship: yes`, `Remote teaching`, `Region: Île-de-France`...

1. **Apriori** finds the **frequent itemsets**: groups of items that appear together in at least a minimum share of the transactions, the **minimal support** σ.
2. From each frequent itemset we build **association rules** of the form *A → B* ("if a programme has A, it often also has B").

**Indicators of a rule A → B**

| Indicator | How it is calculated | How to read it |
|---|---|---|
| **Support** | share of programmes that have A **and** B | how common the pattern is. 0.17 = 17% of all programmes |
| **Confidence** | support(A and B) / support(A) | "when A is true, how often is B true?" 0.99 = 99% of the time |
| **Lift** | confidence / support(B) | > 1: A and B appear together more than by chance. 2 = twice more often than chance. Close to 1: no link |

**Removed items, and why.** An item present in almost every transaction ("taught on site": 93%, "region other than Île-de-France": 83%) produces rules that are always true and say nothing. We keep the informative side only: `Remote teaching` and `Region: Île-de-France`.

**Sankey diagram (lab exercise).** Each rule becomes a flow from its left side (antecedent) to its right side (consequent). The width of the flow is the confidence.

In [9]:
# Function: build_transactions
# Goal: turn each programme of one year into a list of items (a "basket")
# Input: df (DataFrame) clean data; year (int) session to use
# Output: a list of lists of strings, one list per programme
def build_transactions(df, year):
    d = df[df["annee"] == year]
    transactions = []
    for row in d.itertuples(index=False):                       # loop over programmes
        items = [f"Type: {row.type}",                           # main type of programme
                 f"Sector: {row.sector}",                       # public / private
                 "Apprenticeship: yes" if row.is_app else "Apprenticeship: no"]
        if row.remote: items.append("Remote teaching")          # only the informative side is kept
        if row.selective: items.append("Selection tests")
        if row.boarding: items.append("Boarding school")
        if row.adapted_schedule: items.append("Adapted schedules")
        if row.region == "Île-de-France": items.append("Region: Île-de-France")
        transactions.append(items)
    return transactions


# Function: encode_transactions
# Goal: one-hot encoding of the baskets (one column per item, True if the item is in the basket)
# Input: transactions (list of lists) output of build_transactions
# Output: a DataFrame of booleans, the format expected by mlxtend.apriori
def encode_transactions(transactions):
    te = TransactionEncoder()                                   # tool from the lab
    array = te.fit(transactions).transform(transactions)        # list of items -> True/False matrix
    return pd.DataFrame(array, columns=te.columns_)


# Function: support_sensitivity
# Goal: count how many frequent itemsets Apriori finds for several minimal supports (lab exercise 1)
# Input: onehot (DataFrame) output of encode_transactions; supports (list of floats)
# Output: a DataFrame with columns min_support and n_itemsets
def support_sensitivity(onehot, supports):
    counts = [len(apriori(onehot, min_support=s)) for s in supports]   # run Apriori once per threshold
    return pd.DataFrame({"min_support": supports, "n_itemsets": counts})


# Function: mine_rules
# Goal: extract frequent itemsets with Apriori, then association rules, then keep the strong ones
# Input: onehot (DataFrame); min_support (float); min_lift (float); min_confidence (float);
#        max_antecedent (int) maximum number of items on the left side, to keep rules readable
# Output: (itemsets DataFrame, rules DataFrame sorted by lift)
def mine_rules(onehot, min_support=0.02, min_lift=1.2, min_confidence=0.6, max_antecedent=2):
    itemsets = apriori(onehot, min_support=min_support, use_colnames=True)
    try:                                                        # new mlxtend versions need num_itemsets
        rules = association_rules(itemsets, num_itemsets=len(onehot), metric="lift", min_threshold=min_lift)
    except TypeError:                                           # old versions do not have this parameter
        rules = association_rules(itemsets, metric="lift", min_threshold=min_lift)
    rules = rules[(rules["confidence"] >= min_confidence) &
                  (rules["antecedents"].apply(len) <= max_antecedent)].copy()
    rules["rule"] = (rules["antecedents"].apply(lambda s: " + ".join(sorted(s))) + "  →  " +
                     rules["consequents"].apply(lambda s: " + ".join(sorted(s))))
    return itemsets, rules.sort_values("lift", ascending=False).reset_index(drop=True)


# Function: rules_about
# Goal: keep the rules that contain a given item on either side (our research question is apprenticeship)
# Input: rules (DataFrame) output of mine_rules; item (str) e.g. "Apprenticeship: yes"
# Output: the filtered rules DataFrame, still sorted by lift
def rules_about(rules, item):
    mask = rules["antecedents"].apply(lambda s: item in s) | rules["consequents"].apply(lambda s: item in s)
    return rules[mask].reset_index(drop=True)

In [10]:
# Function: plot_patterns
# Goal: draw the results of Query 2: support curve, rules scatter plot, Sankey diagram of the top rules
# Input: sensitivity (DataFrame) output of support_sensitivity; rules (DataFrame) output of mine_rules;
#        top_n (int) number of rules drawn in the Sankey diagram
# Output: a dict of three plotly figures
def plot_patterns(sensitivity, rules, top_n=12):
    fig_s = px.line(sensitivity, x="min_support", y="n_itemsets", markers=True,
                    labels={"min_support": "Minimal support σ", "n_itemsets": "Number of frequent itemsets"},
                    title="Impact of the minimal support on the number of patterns (Apriori)")
    fig_s.update_xaxes(autorange="reversed")                    # from strict (left) to permissive (right)

    fig_r = px.scatter(rules, x="support", y="confidence", color="lift", size="lift",
                       hover_name="rule", color_continuous_scale="Viridis",
                       title=f"{len(rules)} association rules involving apprenticeship (hover a point to read the rule)")

    simple = rules[rules["consequents"].apply(len) == 1]          # one item on the right side: easier to read
    top = simple.drop_duplicates("rule").head(top_n)              # the strongest rules by lift
    left = top["antecedents"].apply(lambda s: " + ".join(sorted(s)))
    right = top["consequents"].apply(lambda s: " + ".join(sorted(s)))
    allnodes = list(dict.fromkeys(list(left) + list(right)))    # distinct nodes, order kept
    source_indices = [allnodes.index(a) for a in left]          # index of each antecedent node
    target_indices = [allnodes.index(c) for c in right]         # index of each consequent node
    conf_values = top["confidence"].round(3).tolist()           # width of each flow
    fig_k = go.Figure(go.Sankey(
        node=dict(label=allnodes, pad=14, thickness=16, color="#4C78A8"),
        link=dict(source=source_indices, target=target_indices, value=conf_values,
                  customdata=top["lift"].round(2), hovertemplate="confidence %{value}<br>lift %{customdata}")))
    fig_k.update_layout(title=f"Top {len(top)} rules involving apprenticeship, by lift (flow width = confidence)", height=560)
    return {"q2_support": fig_s, "q2_rules": fig_r, "q2_sankey": fig_k}

### 5.3 Query 3 (temporal): how fast does apprenticeship grow, and what comes next?

**Technique (lab "Python for analyzing temporal data", lecture "Visualization of temporal data").** We turn the table into a **time series**: one value per year. Then:
- a **Themeriver** (stacked area chart) shows how each type of programme contributes to the total over time;
- a **static state replacement** animation (lecture technique) replays the map year after year: each frame is the state of the system at one date;
- **Prophet** (Facebook's forecasting algorithm, used in the lab) fits a trend and projects it to 2027-2029 with a 95% uncertainty interval.

**Indicators**

| Indicator | How it is calculated | How to read it |
|---|---|---|
| Apprenticeship programmes per year | count of rows with `is_app` True, per `annee` | size of the offer each year |
| **Apprenticeship share (%) per year** | apprenticeship programmes / all programmes x 100, per year | weight of apprenticeship in the whole post-bac offer |
| Growth rate (%) | (value of year N / value of year N-1 - 1) x 100 | speed of growth, negative = decline |
| **New adopters** | institutions whose **first** apprenticeship programme on Parcoursup appears in year N | how many schools start apprenticeship each year (2020 is the first year of the data, so it mixes old and new adopters) |
| Forecast `yhat` and interval [`yhat_lower`, `yhat_upper`] | Prophet prediction and 95% interval | the interval is the range where the true value should fall 95 times out of 100 if the trend continues |
| Back-test error (MAPE, %) | we train on 2020-2024 only, predict 2025-2026, and compare with what really happened: mean of abs(error)/real x 100 | how much we can trust the model; lower is better |

**A limit we state openly.** We only have 7 yearly points. Prophet's seasonal components (weekly, yearly) make no sense with one point per year, so we switch them off and keep the **trend** only. We compare two settings of the parameter `changepoint_prior_scale`, which controls how easily the trend is allowed to bend: 0.05 (default, rigid line) and 0.5 (flexible, able to detect a slowdown).

In [11]:
# Function: build_yearly_series
# Goal: aggregate the data into yearly time series (total, apprenticeship, share, by type, new adopters)
# Input: df (DataFrame) clean data
# Output: a dict with "yearly" (one row per year), "by_type" (year x type) and "adopters" (new institutions per year)
def build_yearly_series(df):
    yearly = df.groupby("annee").agg(programmes=("gta", "count"),
                                     apprenticeship=("is_app", "sum")).reset_index()
    yearly["app_share_%"] = (100 * yearly["apprenticeship"] / yearly["programmes"]).round(1)
    yearly["app_growth_%"] = (100 * yearly["apprenticeship"].pct_change()).round(1)  # year-on-year growth
    app = df[df["is_app"]]
    by_type = app.groupby(["annee", "type"]).size().rename("apprenticeship").reset_index()
    main_types = app[app["annee"] == app["annee"].max()]["type"].value_counts().head(6).index  # 6 largest types
    by_type["type"] = np.where(by_type["type"].isin(main_types), by_type["type"], "Other types")
    by_type = by_type.groupby(["annee", "type"], as_index=False)["apprenticeship"].sum()
    first_year = app.groupby("etab_uai")["annee"].min()            # first year with apprenticeship, per school
    adopters = first_year.value_counts().sort_index().rename_axis("annee").rename("new_adopters").reset_index()
    return {"yearly": yearly, "by_type": by_type, "adopters": adopters}


# Function: forecast_series
# Goal: forecast a yearly series with Prophet (trend only), or with a linear trend if Prophet is not installed
# Input: years (list of int); values (list of float); periods (int) years to forecast;
#        changepoint_prior_scale (float) flexibility of the trend
# Output: a DataFrame with ds (date), yhat (forecast), yhat_lower and yhat_upper (95% interval)
def forecast_series(years, values, periods=3, changepoint_prior_scale=0.05):
    data = pd.DataFrame({"ds": pd.to_datetime([f"{y}-01-01" for y in years]),  # Prophet needs ds and y
                         "y": values})
    if PROPHET_OK:
        model = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False,
                        interval_width=0.95, changepoint_prior_scale=changepoint_prior_scale,
                        n_changepoints=max(0, min(5, len(data) - 2)))
        model.fit(data)                                                     # learn the trend
        future = model.make_future_dataframe(periods=periods, freq="YS")   # add the years to predict
        return model.predict(future)[["ds", "yhat", "yhat_lower", "yhat_upper"]]
    # Fallback: straight line fitted by least squares, interval = +/- 1.96 x residual standard deviation
    slope, intercept = np.polyfit(years, values, 1)
    all_years = list(years) + [years[-1] + k for k in range(1, periods + 1)]
    yhat = intercept + slope * np.array(all_years)
    sd = np.std(np.array(values) - (intercept + slope * np.array(years)))
    return pd.DataFrame({"ds": pd.to_datetime([f"{y}-01-01" for y in all_years]),
                         "yhat": yhat, "yhat_lower": yhat - 1.96 * sd, "yhat_upper": yhat + 1.96 * sd})


# Function: backtest_forecast
# Goal: measure forecast quality by hiding the last years, predicting them and comparing with reality
# Input: years (list of int); values (list of float); tests (list of int) numbers of hidden years; scales (list of float)
# Output: a DataFrame with one row per (hidden years, setting): predictions, real values, MAPE (%)
def backtest_forecast(years, values, tests=(1, 2), scales=(0.05, 0.5)):
    rows = []
    for n_test in tests:
        for cps in scales:
            pred = forecast_series(years[:-n_test], values[:-n_test], periods=n_test, changepoint_prior_scale=cps)
            yhat = pred["yhat"].tail(n_test).values                       # predictions for the hidden years
            real = np.array(values[-n_test:])
            mape = float(np.mean(np.abs(yhat - real) / real) * 100)        # mean absolute percentage error
            rows.append({"trained_on": f"{years[0]}-{years[-n_test - 1]}",
                         "predicted_years": years[-n_test:], "changepoint_prior_scale": cps,
                         "predicted": np.round(yhat).astype(int).tolist(),
                         "real": [int(v) for v in real], "MAPE_%": round(mape, 1)})
    return pd.DataFrame(rows)

In [12]:
# Function: plot_temporal
# Goal: draw the results of Query 3: Themeriver, adopters, forecasts, animated map (static state replacement)
# Input: series (dict) output of build_yearly_series; forecasts (dict name -> forecast DataFrame);
#        df (DataFrame) clean data for the animated map
# Output: a dict of four plotly figures
def plot_temporal(series, forecasts, df):
    bt = series["by_type"]
    fig_river = go.Figure()
    for t in bt.groupby("type")["apprenticeship"].sum().sort_values(ascending=False).index:  # biggest first
        s = bt[bt["type"] == t]
        fig_river.add_trace(go.Scatter(x=s["annee"], y=s["apprenticeship"], name=t, mode="lines",
                                       stackgroup="one", line=dict(width=0.5)))  # stacked areas = Themeriver
    fig_river.update_layout(title="Themeriver: apprenticeship programmes per year, by type",
                            xaxis_title="Parcoursup session", yaxis_title="Apprenticeship programmes", height=480)

    ad = series["adopters"]
    fig_ad = px.bar(ad, x="annee", y="new_adopters", text="new_adopters",
                    labels={"annee": "Year of the first apprenticeship programme", "new_adopters": "Institutions"},
                    title="New adopters: institutions opening their first apprenticeship programme (2020 includes earlier adopters)")

    y = series["yearly"]
    fig_fc = go.Figure()
    fig_fc.add_trace(go.Scatter(x=y["annee"], y=y["apprenticeship"], mode="lines+markers",
                                name="Observed", line=dict(color="black", width=3)))
    colors = {"Rigid trend (cps 0.05)": "#E45756", "Flexible trend (cps 0.5)": "#4C78A8"}
    for name, fc in forecasts.items():
        yrs = fc["ds"].dt.year
        fig_fc.add_trace(go.Scatter(x=list(yrs) + list(yrs[::-1]),
                                    y=list(fc["yhat_upper"]) + list(fc["yhat_lower"][::-1]),
                                    fill="toself", fillcolor=colors.get(name, "#999"), opacity=0.15,
                                    line=dict(width=0), showlegend=False, hoverinfo="skip"))
        fig_fc.add_trace(go.Scatter(x=yrs, y=fc["yhat"], mode="lines", name=name,
                                    line=dict(dash="dash", color=colors.get(name, "#999"))))
    fig_fc.update_layout(title="Forecast of apprenticeship programmes with Prophet (shaded = 95% interval)",
                         xaxis_title="Year", yaxis_title="Apprenticeship programmes", height=480)

    sites = (df[df["is_app"] & df["metropolitan"]]
             .groupby(["annee", "etab_uai"]).agg(lat=("lat", "first"), lon=("lon", "first"),
                                                 programmes=("gta", "count"),
                                                 institution=("institution", "first")).reset_index())
    fig_anim = px.scatter_map(sites.sort_values("annee"), lat="lat", lon="lon", size="programmes",
                              animation_frame="annee", hover_name="institution", size_max=14,
                              zoom=4.3, center=dict(lat=46.6, lon=2.4), opacity=0.6,
                              color_discrete_sequence=["#1f5fa8"],
                              title="Static state replacement: institutions offering apprenticeship, year by year")
    fig_anim.update_layout(map_style="open-street-map", height=600, margin=dict(l=0, r=0, t=50, b=0))
    return {"q3_river": fig_river, "q3_adopters": fig_ad, "q3_forecast": fig_fc, "q3_animation": fig_anim}

### 5.4 Query 4 (spatial): where are the hubs of apprenticeship?

**From rows to places.** Clustering works on places, so we first build one row per **institution** (site) offering apprenticeship in 2026, with its coordinates and its number of apprenticeship programmes.

**Technique 1: OPTICS (lab "Python for spatial information").** OPTICS is a **density-based** clustering algorithm: a cluster is a zone where points are close to many other points. Points in empty zones are labelled **-1 = noise** (isolated sites). Unlike K-means, we do not choose the number of clusters, and clusters can have any shape.
- `metric="haversine"`: the distance between two GPS points along the surface of the Earth. Latitude and longitude are angles, not kilometres, so a plain Euclidean distance would be wrong. The haversine metric needs angles in **radians**, so we convert, and we divide a distance in km by the Earth radius (6,371 km) to express it in radians.
- `min_samples`: minimum number of neighbours for a point to be in a dense zone.
- `eps` (with `cluster_method="dbscan"`): radius of the neighbourhood. At national scale we use 10 km (the size of a city and its suburbs).

**Technique 2: K-means (lab exercise).** K-means splits the points into exactly *k* groups, each around a centre (centroid). It forces every point into a group (no noise). We choose *k* with the **silhouette score**: for each point, it compares the distance to its own group with the distance to the nearest other group. It goes from -1 (wrong group) to 1 (perfect group); we keep the *k* with the highest average.

We apply OPTICS to the whole of mainland France (to find the national hubs) and both algorithms to the Paris region (Île-de-France), the densest area, to compare them.

**Indicators of each cluster**

| Indicator | How it is calculated | How to read it |
|---|---|---|
| Sites | number of institutions in the cluster | how many places offer apprenticeship there |
| Apprenticeship programmes | sum of the programmes of its sites | size of the local offer |
| Main town | most frequent `commune` in the cluster | name we give to the hub |
| Radius (km) | mean haversine distance from the sites to the cluster centre | how compact the hub is |
| Noise share (%) | sites labelled -1 / all sites x 100 | share of isolated sites, far from any hub |

In [13]:
EARTH_RADIUS_KM = 6371.0   # used to convert kilometres into radians for the haversine metric

# Function: build_sites
# Goal: one row per institution offering apprenticeship in a given year (mainland France)
# Input: df (DataFrame) clean data; year (int); region (str or None) to restrict to one region
# Output: a DataFrame with etab_uai, institution, commune, lat, lon, programmes, main_type, sector
def build_sites(df, year, region=None):
    d = df[(df["annee"] == year) & df["is_app"] & df["metropolitan"]]
    if region is not None:
        d = d[d["region"] == region]
    sites = d.groupby("etab_uai").agg(institution=("institution", "first"), commune=("commune", "first"),
                                      region=("region", "first"), lat=("lat", "first"), lon=("lon", "first"),
                                      programmes=("gta", "count"),
                                      main_type=("type", lambda s: s.value_counts().index[0]),
                                      sector=("sector", "first")).reset_index()
    return sites


# Function: haversine_km
# Goal: distance in km between points and one reference point, along the surface of the Earth
# Input: lat, lon (arrays, degrees); lat0, lon0 (floats, degrees)
# Output: numpy array of distances in km
def haversine_km(lat, lon, lat0, lon0):
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, lat0, lon0))
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


# Function: run_optics
# Goal: density-based spatial clustering with OPTICS and the haversine distance
# Input: sites (DataFrame with lat, lon); min_samples (int); eps_km (float or None):
#        if eps_km is given, clusters are cut at this radius (cluster_method "dbscan"),
#        otherwise OPTICS uses its own "xi" method (steepness of the reachability plot)
# Output: (numpy array of labels, fitted OPTICS model); label -1 means noise
def run_optics(sites, min_samples=15, eps_km=None, xi=0.05):
    coords = np.radians(sites[["lat", "lon"]].values)          # haversine works with radians
    if eps_km is not None:
        model = OPTICS(min_samples=min_samples, metric="haversine", cluster_method="dbscan",
                       eps=eps_km / EARTH_RADIUS_KM, max_eps=3 * eps_km / EARTH_RADIUS_KM)
    else:
        model = OPTICS(min_samples=min_samples, metric="haversine", xi=xi, min_cluster_size=10)
    labels = model.fit_predict(coords)
    return labels, model


# Function: run_kmeans
# Goal: K-means on the sites, choosing k with the best silhouette score
# Input: sites (DataFrame with lat, lon); k_values (iterable of int); seed (int) for reproducibility
# Output: (labels of the best k, best k, DataFrame of silhouette scores)
def run_kmeans(sites, k_values=range(3, 13), seed=42):
    # Near Paris, 1 degree of longitude is about 0.66 x 1 degree of latitude, so we rescale the
    # longitude to make the Euclidean distance used by K-means close to real kilometres
    X = np.column_stack([sites["lat"], sites["lon"] * np.cos(np.radians(sites["lat"].mean()))])
    scores = []
    for k in k_values:
        labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
        scores.append({"k": k, "silhouette": round(silhouette_score(X, labels), 3)})
    scores = pd.DataFrame(scores)
    best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
    best_labels = KMeans(n_clusters=best_k, n_init=10, random_state=seed).fit_predict(X)
    return best_labels, best_k, scores


# Function: summarize_clusters
# Goal: compute the indicators of each cluster (sites, programmes, main town, centre, radius)
# Input: sites (DataFrame); label_col (str) column holding the cluster labels
# Output: a DataFrame with one row per cluster (noise excluded), biggest first
def summarize_clusters(sites, label_col):
    rows = []
    for lab, g in sites[sites[label_col] != -1].groupby(label_col):
        clat, clon = g["lat"].mean(), g["lon"].mean()                 # centre of the cluster
        rows.append({label_col: lab, "sites": len(g), "programmes": int(g["programmes"].sum()),
                     "main_town": g["commune"].value_counts().index[0],
                     "centre_lat": round(clat, 4), "centre_lon": round(clon, 4),
                     "radius_km": round(haversine_km(g["lat"], g["lon"], clat, clon).mean(), 1)})
    out = pd.DataFrame(rows).sort_values("programmes", ascending=False).reset_index(drop=True)
    noise = 100 * (sites[label_col] == -1).mean()
    print(f"{label_col}: {len(out)} clusters, noise = {noise:.1f}% of the sites")
    return out

In [14]:
# Function: plot_clusters_map
# Goal: map the sites coloured by cluster, on an OpenStreetMap background (lab technique)
# Input: sites (DataFrame); label_col (str); title (str); zoom (float); center (dict lat/lon); names (dict label->name)
# Output: a plotly figure
def plot_clusters_map(sites, label_col, title, zoom=4.3, center=None, names=None):
    s = sites.copy()
    s["cluster"] = s[label_col].map(names) if names else s[label_col].astype(str)
    s.loc[s[label_col] == -1, "cluster"] = "Isolated (noise)"
    s["cluster"] = s["cluster"].fillna(s[label_col].astype(str))
    fig = px.scatter_map(s, lat="lat", lon="lon", color="cluster", size="programmes", size_max=12,
                         hover_name="institution", hover_data={"commune": True, "programmes": True,
                                                               "lat": False, "lon": False},
                         zoom=zoom, center=center or dict(lat=46.6, lon=2.4), opacity=0.75, title=title)
    fig.update_layout(map_style="open-street-map", height=620, margin=dict(l=0, r=0, t=50, b=0),
                      legend=dict(title="", font=dict(size=10)))
    for tr in fig.data:                                           # noise drawn in light grey
        if tr.name == "Isolated (noise)":
            tr.marker.color = "#BBBBBB"
    return fig


# Function: plot_reachability
# Goal: reachability plot of OPTICS (valleys = clusters, peaks = gaps between clusters)
# Input: model (fitted OPTICS); labels (array); title (str)
# Output: a plotly figure
def plot_reachability(model, labels, title):
    order = model.ordering_                                       # order in which OPTICS visited the points
    reach = model.reachability_[order] * EARTH_RADIUS_KM          # radians -> km
    reach[np.isinf(reach)] = np.nan                               # the first point has no reachability
    lab = np.where(labels[order] == -1, "noise", "cluster " + labels[order].astype(str))
    fig = px.bar(x=np.arange(len(order)), y=reach, color=lab,
                 labels={"x": "Sites in OPTICS order", "y": "Reachability distance (km)"}, title=title)
    fig.update_layout(showlegend=False, height=380, bargap=0)
    return fig


# Function: plot_silhouette
# Goal: silhouette score for each k tested with K-means
# Input: scores (DataFrame with k and silhouette)
# Output: a plotly figure
def plot_silhouette(scores):
    return px.line(scores, x="k", y="silhouette", markers=True,
                   title="K-means: silhouette score by number of clusters k (higher is better)")

### 5.5 Extra point: a second dataset to interpret the results (InserJeunes)

**What is it?** *InserJeunes* is the official system that follows apprentices after their training. For each training centre (identified by its **UAI** code, the same code as in Parcoursup) it publishes:

| Indicator (from the source) | Meaning |
|---|---|
| `taux_contrats_interrompus` | **% of apprenticeship contracts interrupted** before the end of the training (published for the 2018-2019 cohorts) |
| `taux_emploi_6_mois` | % of former apprentices in salaried employment 6 months after leaving |
| `taux_poursuite_etudes` | % who continue studying |
| `part_des_autres_situations` | % who are neither employed nor studying 6 months later |

Each indicator covers two cohorts together ("cumul 2023-2024" = students who left in 2023 and 2024), which makes small centres more reliable. For each centre we keep the most recent value available for every indicator.

**How we join the two datasets.** A **join** (like in SQL) puts side by side the rows of two tables that share a key. Our key is the UAI code. We use a *left join*: we keep all Parcoursup sites and add the InserJeunes columns when the centre exists in InserJeunes.

**Indicators.** For each OPTICS hub (Query 4) we compute the **average interruption rate** and the **average employment rate at 6 months** of its centres. A hub with many sites *and* a high interruption rate is a place where many apprentices break their contract before the end.

In [15]:
# Function: clean_inserjeunes
# Goal: keep, for each training centre (UAI), the most recent value of each outcome indicator
# Input: ij (DataFrame) raw InserJeunes table
# Output: a DataFrame with one row per UAI and the outcome columns
def clean_inserjeunes(ij):
    ij = ij.copy()
    ij["end_year"] = ij["annee"].str[-4:].astype(int)            # "cumul 2023-2024" -> 2024
    cols = ["taux_contrats_interrompus", "taux_emploi_6_mois", "taux_poursuite_etudes",
            "part_des_autres_situations"]
    for c in cols:
        ij[c] = pd.to_numeric(ij[c], errors="coerce")             # text -> number, empty -> NaN
    ij = ij.sort_values("end_year")
    out = ij.groupby("uai")[cols].last()                          # .last() skips NaN: latest non-empty value
    out["interruption_cohort"] = (ij.dropna(subset=["taux_contrats_interrompus"])
                                    .groupby("uai")["annee"].last())
    return out.reset_index().rename(columns={"uai": "etab_uai"})


# Function: enrich_sites
# Goal: left join of the Parcoursup sites with the InserJeunes outcomes (key = UAI)
# Input: sites (DataFrame) output of build_sites; outcomes (DataFrame) output of clean_inserjeunes
# Output: the sites DataFrame with the outcome columns added (NaN when the centre is not in InserJeunes)
def enrich_sites(sites, outcomes):
    merged = sites.merge(outcomes, on="etab_uai", how="left")
    found = merged["taux_emploi_6_mois"].notna().sum()
    found_i = merged["taux_contrats_interrompus"].notna().sum()
    print(f"{found} of {len(sites)} sites found in InserJeunes (employment rate), "
          f"{found_i} with an interruption rate")
    return merged


# Function: outcomes_by_cluster
# Goal: average outcomes of the training centres of each spatial cluster
# Input: enriched (DataFrame) output of enrich_sites; label_col (str); clusters (DataFrame) output of summarize_clusters
# Output: the clusters DataFrame with average interruption and employment rates
def outcomes_by_cluster(enriched, label_col, clusters):
    agg = (enriched[enriched[label_col] != -1].groupby(label_col)
           .agg(interruption_rate_mean=("taux_contrats_interrompus", "mean"),
                centres_with_interruption_data=("taux_contrats_interrompus", "count"),
                employment_6m_mean=("taux_emploi_6_mois", "mean"))
           .round(1).reset_index())
    return clusters.merge(agg, on=label_col, how="left")


# Function: plot_external
# Goal: draw the outcomes: histogram of interruption rates and outcomes of the biggest hubs
# Input: enriched (DataFrame); hubs (DataFrame) output of outcomes_by_cluster;
#        min_centres (int) a hub is drawn only if at least this many of its centres have an interruption rate
# Output: a dict of two plotly figures
def plot_external(enriched, hubs, min_centres=5):
    e = enriched.dropna(subset=["taux_contrats_interrompus"])
    fig_h = px.histogram(e, x="taux_contrats_interrompus", nbins=25,
                         labels={"taux_contrats_interrompus": "Interrupted contracts (%)"},
                         title=f"Share of apprenticeship contracts interrupted, {len(e)} training centres (InserJeunes)")
    fig_h.add_vline(x=e["taux_contrats_interrompus"].median(), line_dash="dash",
                    annotation_text=f"median {e['taux_contrats_interrompus'].median():.0f}%")
    h = hubs[hubs["centres_with_interruption_data"] >= min_centres]   # averages on 1-2 centres are not reliable
    fig_b = px.scatter(h, x="employment_6m_mean", y="interruption_rate_mean", size="programmes",
                       text="main_town", size_max=40,
                       labels={"employment_6m_mean": "Employment 6 months after (%)",
                               "interruption_rate_mean": "Interrupted contracts (%)"},
                       title=f"Hubs with at least {min_centres} centres measured: interrupted contracts vs employment (bubble = programmes)")
    fig_b.update_traces(textposition="top center")
    return {"ext_hist": fig_h, "ext_hubs": fig_b}

## 6. Visualization dashboard (Dash)

**Dash** builds a small web application in Python. The page is described as a tree of components (`html.H1` for a title, `dcc.Graph` for a plotly chart, `dcc.Tabs` for tabs). Here we reuse the figures created above, so the dashboard and the notebook always show the same numbers.

The dashboard shows the team members and the dataset names in its header (project requirement), five key figures (KPIs), then one tab per query. Because a Dash app needs a running Python server, we also save a **static HTML copy** (`Collin_dashboard.html`) that opens in any browser, as evidence that it works.

In [16]:
# Function: build_kpis
# Goal: compute the headline numbers shown at the top of the dashboard
# Input: series (dict) output of build_yearly_series; q1 (dict); sites_fr (DataFrame); enriched (DataFrame)
# Output: a list of (label, value) tuples
def build_kpis(series, q1, sites_fr, enriched):
    y = series["yearly"].set_index("annee")
    first, last = y.index.min(), y.index.max()
    interr = enriched["taux_contrats_interrompus"].median()
    return [(f"Apprenticeship programmes {last}", f"{int(y.loc[last, 'apprenticeship']):,}"),
            (f"Growth {first}-{last}", f"x{y.loc[last, 'apprenticeship'] / y.loc[first, 'apprenticeship']:.1f}"),
            (f"Share of the post-bac offer {last}", f"{y.loc[last, 'app_share_%']:.0f}%"),
            (f"Institutions offering it {last}", f"{len(sites_fr):,}"),
            ("Median share of contracts interrupted", f"{interr:.0f}%")]


# Function: build_dashboard
# Goal: assemble the Dash application (header with team and datasets, KPIs, one tab per query)
# Input: figs (dict name -> plotly figure); kpis (list of tuples); team (list of str)
# Output: a Dash app object (not started yet)
def build_dashboard(figs, kpis, team):
    card = {"border": "1px solid #d0d7de", "borderRadius": "6px", "padding": "10px 16px",
            "minWidth": "170px", "background": "#f6f8fa"}
    def graph(name):                                            # one chart = one dcc.Graph
        return dcc.Graph(figure=figs[name], style={"marginBottom": "18px"})
    app = Dash(__name__)
    app.title = "Apprenticeship in French higher education"
    app.layout = html.Div(style={"fontFamily": "Georgia, serif", "maxWidth": "1200px", "margin": "auto"}, children=[
        html.H1("The rise of apprenticeship in French higher education, 2020-2026"),
        html.P("Team: " + ", ".join(team)),
        html.P("Datasets: Cartographie des formations Parcoursup 2020-2026 (MESR open data) and "
               "InserJeunes apprenticeship by CFA (DEPP/DARES open data)"),
        html.Div([html.Div([html.Div(v, style={"fontSize": "26px", "fontWeight": "bold"}),
                            html.Div(k, style={"fontSize": "13px"})], style=card) for k, v in kpis],
                 style={"display": "flex", "gap": "12px", "flexWrap": "wrap", "margin": "12px 0 18px"}),
        dcc.Tabs([
            dcc.Tab(label="Q1 Grouping", children=[graph("q1_types"), graph("q1_regions")]),
            dcc.Tab(label="Q2 Pattern mining", children=[graph("q2_sankey"), graph("q2_rules"), graph("q2_support")]),
            dcc.Tab(label="Q3 Temporal", children=[graph("q3_forecast"), graph("q3_river"),
                                                   graph("q3_adopters"), graph("q3_animation")]),
            dcc.Tab(label="Q4 Spatial", children=[graph("q4_france"), graph("q4_idf_optics"),
                                                  graph("q4_idf_kmeans"), graph("q4_reach")]),
            dcc.Tab(label="Second dataset", children=[graph("ext_hist"), graph("ext_hubs")]),
        ]),
    ])
    return app


# Function: export_dashboard_html
# Goal: save a static HTML copy of the dashboard (same header, KPIs and charts), evidence that it works
# Input: figs (dict); kpis (list of tuples); team (list of str); path (str) output file
# Output: the path of the written file
def export_dashboard_html(figs, kpis, team, path):
    sections = [("Q1 Grouping", ["q1_types", "q1_regions"]),
                ("Q2 Pattern mining", ["q2_sankey", "q2_rules", "q2_support"]),
                ("Q3 Temporal", ["q3_forecast", "q3_river", "q3_adopters", "q3_animation"]),
                ("Q4 Spatial", ["q4_france", "q4_idf_optics", "q4_idf_kmeans", "q4_reach"]),
                ("Second dataset", ["ext_hist", "ext_hubs"])]
    kpi_html = "".join(f"<div class='kpi'><b>{v}</b><span>{k}</span></div>" for k, v in kpis)
    body, first = [], True
    for title, names in sections:
        body.append(f"<h2>{title}</h2>")
        for n in names:                                         # plotly.js is embedded once (offline file)
            body.append(figs[n].to_html(full_html=False, include_plotlyjs=True if first else False))
            first = False
    html_page = f'''<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<title>Dashboard: apprenticeship in French higher education</title>
<style>body{{font-family:Georgia,serif;max-width:1200px;margin:auto;padding:16px;color:#1b1f24}}
.kpis{{display:flex;gap:12px;flex-wrap:wrap}} .kpi{{border:1px solid #d0d7de;border-radius:6px;padding:10px 16px;background:#f6f8fa}}
.kpi b{{display:block;font-size:26px}} .kpi span{{font-size:13px}} h2{{border-bottom:2px solid #1f5fa8;padding-bottom:4px;margin-top:36px}}</style>
</head><body><h1>The rise of apprenticeship in French higher education, 2020-2026</h1>
<p>Team: {", ".join(team)}</p>
<p>Datasets: Cartographie des formations Parcoursup 2020-2026 (MESR open data) and InserJeunes apprenticeship by CFA (DEPP/DARES open data)</p>
<p><i>Static export of the Dash dashboard generated by the notebook.</i></p>
<div class="kpis">{kpi_html}</div>{"".join(body)}</body></html>'''
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_page)
    print("Static dashboard saved to", path)
    return path

## 7. Main function

`main()` runs the whole KDD process in order and returns every result in a dictionary called `results`, so that we can comment them one by one in the next section. At the end it launches the dashboard.

- In Jupyter, the dashboard opens at **http://127.0.0.1:8050** (click the link printed below the cell).
- If the port is busy, change `DASHBOARD_PORT` in the settings cell.

In [17]:
# Function: main
# Goal: run the full project: load, explore, clean, four queries, second dataset, dashboard
# Input: none (uses the settings defined at the top of the notebook)
# Output: a dict "results" with all tables and figures
def main():
    results, figs = {}, {}

    # Step 1: data collection and exploration
    raw = load_data(MAIN_FILE, MAIN_URL)
    results["raw_shape"] = raw.shape
    results["dictionary"] = summarize_data(raw)

    # Steps 2-3: pre-processing and transformation
    df = clean_parcoursup(raw)
    results["df"] = df

    # Step 4, Query 1: grouping
    q1 = query_grouping(df, FOCUS_YEAR)
    results["q1"] = q1
    figs.update(plot_grouping(q1, FOCUS_YEAR))

    # Step 4, Query 2: frequent patterns and association rules
    transactions = build_transactions(df, FOCUS_YEAR)
    onehot = encode_transactions(transactions)
    results["item_frequency"] = onehot.mean().sort_values(ascending=False).round(3)
    sensitivity = support_sensitivity(onehot, [0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05, 0.03, 0.02, 0.01])
    itemsets, rules_all = mine_rules(onehot, min_support=0.02, min_lift=1.2, min_confidence=0.6)
    rules = rules_about(rules_all, "Apprenticeship: yes")
    results.update({"sensitivity": sensitivity, "itemsets": itemsets, "rules_all": rules_all,
                    "rules": rules, "n_transactions": len(transactions)})
    figs.update(plot_patterns(sensitivity, rules))

    # Step 4, Query 3: time series and forecasting
    series = build_yearly_series(df)
    yearly = series["yearly"]
    years, values = yearly["annee"].tolist(), yearly["apprenticeship"].astype(float).tolist()
    forecasts = {"Rigid trend (cps 0.05)": forecast_series(years, values, FORECAST_YEARS, 0.05),
                 "Flexible trend (cps 0.5)": forecast_series(years, values, FORECAST_YEARS, 0.5)}
    share_forecast = forecast_series(years, yearly["app_share_%"].tolist(), FORECAST_YEARS, 0.5)
    backtest = backtest_forecast(years, values, tests=(1, 2))
    results.update({"series": series, "forecasts": forecasts, "share_forecast": share_forecast,
                    "backtest": backtest})
    figs.update(plot_temporal(series, forecasts, df))

    # Step 4, Query 4: spatial clustering
    sites_fr = build_sites(df, FOCUS_YEAR)                               # mainland France
    sites_fr["hub"], optics_fr = run_optics(sites_fr, min_samples=15, eps_km=10)
    hubs = summarize_clusters(sites_fr, "hub")
    hub_names = dict(zip(hubs["hub"], hubs["main_town"] + " hub"))
    sites_idf = build_sites(df, FOCUS_YEAR, region="Île-de-France")      # Paris region zoom
    sites_idf["optics"], optics_idf = run_optics(sites_idf, min_samples=8, xi=0.03)
    sites_idf["kmeans"], best_k, sil = run_kmeans(sites_idf, range(3, 13))
    idf_optics = summarize_clusters(sites_idf, "optics")
    idf_kmeans = summarize_clusters(sites_idf, "kmeans")
    results.update({"sites_fr": sites_fr, "hubs": hubs, "sites_idf": sites_idf, "best_k": best_k,
                    "silhouette": sil, "idf_optics": idf_optics, "idf_kmeans": idf_kmeans})
    paris = dict(lat=48.86, lon=2.35)
    figs["q4_france"] = plot_clusters_map(sites_fr, "hub", "OPTICS hubs of apprenticeship, mainland France (eps 10 km)",
                                          names=hub_names)
    figs["q4_idf_optics"] = plot_clusters_map(sites_idf, "optics", "Paris region: OPTICS dense pockets",
                                              zoom=8.6, center=paris)
    figs["q4_idf_kmeans"] = plot_clusters_map(sites_idf, "kmeans", f"Paris region: K-means territories (k = {best_k})",
                                              zoom=8.6, center=paris)
    figs["q4_reach"] = plot_reachability(optics_idf, sites_idf["optics"].values,
                                         "OPTICS reachability plot, Paris region (valleys = clusters)")
    figs["q4_silhouette"] = plot_silhouette(sil)

    # Extra point: second dataset
    ij_raw = load_data(EXT_FILE, EXT_URL)
    outcomes = clean_inserjeunes(ij_raw)
    enriched = enrich_sites(sites_fr, outcomes)
    hubs_out = outcomes_by_cluster(enriched, "hub", hubs)
    results.update({"ij_raw_shape": ij_raw.shape, "enriched": enriched, "hubs_outcomes": hubs_out})
    figs.update(plot_external(enriched, hubs_out))

    # Step 5: dashboard
    kpis = build_kpis(series, q1, sites_fr, enriched)
    results.update({"figs": figs, "kpis": kpis})
    export_dashboard_html(figs, kpis, TEAM, "Collin_dashboard.html")
    if RUN_DASHBOARD:
        app = build_dashboard(figs, kpis, TEAM)
        results["app"] = app
        app.run(jupyter_mode="external", port=DASHBOARD_PORT, debug=False)   # non-blocking in Jupyter
    return results


# Main block
if __name__ == "__main__":
    results = main()

Loaded 157,497 rows x 25 columns from data/fr-esr-cartographie_formations_parcoursup.csv.gz
Clean table: 157,497 rows, 7,686 institutions, 60,548 apprenticeship rows
hub: 49 clusters, noise = 43.9% of the sites
optics: 15 clusters, noise = 56.7% of the sites
kmeans: 4 clusters, noise = 0.0% of the sites
Loaded 11,314 rows x 15 columns from data/fr-en-inserjeunes-cfa.csv
956 of 3743 sites found in InserJeunes (employment rate), 347 with an interruption rate
Static dashboard saved to Collin_dashboard.html
Dash app running on http://127.0.0.1:8050/


Clean table: 157,497 rows, 7,686 institutions, 60,548 apprenticeship rows


hub: 49 clusters, noise = 43.9% of the sites


optics: 15 clusters, noise = 56.7% of the sites
kmeans: 4 clusters, noise = 0.0% of the sites


Loaded 11,314 rows x 15 columns from data/fr-en-inserjeunes-cfa.csv
956 of 3743 sites found in InserJeunes (employment rate), 347 with an interruption rate
Static dashboard saved to Collin_dashboard.html


Dash app running on http://127.0.0.1:8050/


## 8. Results and interpretation (step 5 of KDD)

The cells below only **display** what `main()` computed. The comments are written from the numbers obtained when we ran the notebook on the data downloaded on 25 November 2026 (the files are updated by the ministries, so a later download can change the last digits).

### 8.1 Exploration of the raw data

In [18]:
print("Rows, columns of the raw table:", results["raw_shape"])
results["dictionary"]

Rows, columns of the raw table: (157497, 25)


,column,type,missing,missing_%,distinct_values,range_or_examples
0,annee,int64,0,0.0,7,"[2020 , 2026]"
1,etab_uai,object,0,0.0,7686,0673021V | 0755976N | 0751719L
2,etab_nom,object,0,0.0,11755,Université de Lille (59) | Université de Strasbourg (67) | Université Paris Nanterre (92)
3,tc,object,0,0.0,6,Publics | Privés | Privés sous contrat
4,tf,object,0,0.0,86,BTS - BTSA - BTSM | Licence | CPGE
5,nm,object,0,0.0,11516,BTS - Services - Management Commercial O | BTS - Services - Négociation et digitali | ...
6,fl,object,0,0.0,1928,BTS - Management Commercial Opérationnel | BTS - Négociation et digitalisation de l | ...
7,app,object,96949,61.6,1,Formations en apprentissage
8,int,object,152282,96.7,3,Etablissements avec internat pour filles | Etablissements avec internat pour filles | ...
9,amg,object,0,0.0,24,"Enseignement en présentiel | Enseignement en présentiel,Formations av | Enseignement p..."


**Reading.** 157,497 rows and 25 columns. Only `annee`, `gta`, `gti`, `rnd` and `code_formation` are numbers; everything else is text. The missing values are not errors:
- `app` is empty for 62% of rows: these programmes are simply **not** in apprenticeship;
- `int` (boarding) and `aut` (admission conditions) are only filled when the feature exists;
- `region` is empty for 298 rows: campuses abroad (e.g. Singapore, Barcelona, Hanoi) and a few schools in French Polynesia;
- `rnd` and `code_formation` are empty for exactly the 16,990 rows of 2020, a sign that these codes were created in 2021 (another reason to drop them).

In [19]:
df = results["df"]
df[["annee", "institution", "type", "sector", "is_app", "remote", "selective", "region", "lat", "lon"]].head()

,annee,institution,type,sector,is_app,remote,selective,region,lat,lon
0,2026,La Prépa des INP - Groupe INP - Saint-Denis de la Réunion (974),Engineering schools,Public,False,False,True,La Réunion,-20.8907,55.46790
1,2026,Télécom Saint-Etienne (42),Engineering schools,Public,False,False,True,Auvergne-Rhône-Alpes,45.4535,4.38715
2,2026,Polytech Annecy-Chambéry – site d’Annecy (74),Engineering schools,Public,False,False,True,Auvergne-Rhône-Alpes,45.9194,6.15779
3,2026,Polytech Sorbonne (Paris 5e Arrondissement - 75),Engineering schools,Public,False,False,True,Île-de-France,48.8451,2.35695
4,2026,UTC Université de Technologie de Compiègne (60),Engineering schools,Public,False,False,True,Hauts-de-France,49.4155,2.81939


### 8.2 Query 1: grouping

In [20]:
results["figs"]["q1_types"].show()
results["q1"]["by_type"].head(10)

,type,programmes,apprenticeship,institutions_with_app,app_share_%
0,BTS (2-year technician degree),13966,9142,3230,65.5
1,Specialisation certificates,1103,661,456,59.9
2,Professional certifications,647,645,397,99.7
3,Social work diplomas,706,204,89,28.9
4,Health studies,1449,149,142,10.3
5,BUT (3-year technology bachelor),953,135,76,14.2
6,DEUST (2-year university diploma),117,103,98,88.0
7,DCG (accounting),194,97,97,50.0
8,Sport professions,392,83,66,21.2
9,Art and design,687,9,7,1.3


**Reading.** In 2026, **BTS** programmes make up the bulk of apprenticeship on Parcoursup (9,142 of the 11,251 apprenticeship programmes, 81%) and 65.5% of all BTS can be followed as an apprentice. Some smaller types are almost entirely in apprenticeship (professional certifications, DEUST). At the opposite, business schools and university licences recruit almost no apprentices through Parcoursup: their apprenticeship tracks usually start later (third year, master) and are not listed there.

In [21]:
results["figs"]["q1_regions"].show()
results["q1"]["top_institutions"]

,institution,commune,sector,apprenticeship_programmes
0,AURLOM BTS+ (Paris 11e Arrondissement - 75),Paris,"Private, no State contract",30
1,Organisme de formation-Centre de formation d'apprentis - IFFP - Institut Francais de F...,Puteaux,"Private, no State contract",22
2,CFA E2SE Ecole Supérieure des Services aux Entreprises (Caen - 14),Caen,"Private, no State contract",18
3,CFA de la pharmacie de Champagne méridionale (Saint-Julien-les-Villas - 10),Saint-Julien-les-Villas,"Private, State contract",17
4,CFA du Groupe AFTEC - Rennes (35),Rennes,"Private, no State contract",17
5,ISA - Campus EDUCTIVE LYON (Sciences-U Lyon) (Lyon 3e Arrondissement - 69),Lyon,"Private, no State contract",17
6,ALMEA FORMATIONS INTERPRO (Châlons-en-Champagne - 51),Châlons-en-Champagne,"Private, no State contract",16
7,ALMEA FORMATIONS INTERPRO (Charleville-Mézières - 08),Charleville-Mézières,"Private, no State contract",16
8,Ecole de Commerce Pigier Nantes (Saint-Herblain - 44),Saint-Herblain,"Private, no State contract",16
9,ESICAD (Montpellier - 34),Montpellier,"Private, no State contract",16


**Reading.** Auvergne-Rhône-Alpes (1,551) and Île-de-France (1,550) are tied for the largest apprenticeship offer. Île-de-France has, however, one of the **lowest shares** in mainland France (36%, against 45 to 50% elsewhere): its offer also contains many university and selective programmes, which are rarely in apprenticeship. Most overseas territories have much lower shares (17 to 37%). The ranking of institutions shows that the largest providers are **private training centres**, often part of networks with several campuses (the same brand appears in several towns). This is consistent with the 2018 reform, which let private organisations open apprenticeship training centres freely.

### 8.3 Query 2: frequent patterns and association rules

In [22]:
print("Number of transactions (programmes in 2026):", results["n_transactions"])
print("\nShare of transactions containing each item:")
print(results["item_frequency"].head(12))
results["figs"]["q2_support"].show()
results["sensitivity"]

Number of transactions (programmes in 2026): 25793

Share of transactions containing each item:
Apprenticeship: no                      0.564
Sector: Public                          0.562
Type: BTS (2-year technician degree)    0.541
Apprenticeship: yes                     0.436
Adapted schedules                       0.289
Sector: Private, no State contract      0.225
Sector: Private, State contract         0.213
Region: Île-de-France                   0.166
Selection tests                         0.155
Type: Licence (university bachelor)     0.088
Remote teaching                         0.072
Type: Health studies                    0.056
dtype: float64


,min_support,n_itemsets
0,0.50,3
1,0.40,5
2,0.30,6
3,0.20,13
4,0.15,19
5,0.10,27
6,0.05,51
7,0.03,89
8,0.02,148
9,0.01,224


**Reading of the support curve (lab exercise 1).** With a strict minimal support (0.5) only a handful of itemsets are frequent; when the threshold goes down, the number of itemsets explodes (several hundred at 0.02). This is the **anti-monotony** property used by Apriori: an itemset cannot be more frequent than any of its parts, so lowering σ lets longer and rarer combinations in. We chose σ = 0.02 (2% of 25,793 programmes, about 500 programmes): rare enough to keep specific combinations, frequent enough to be meaningful.

In [23]:
results["figs"]["q2_sankey"].show()
results["figs"]["q2_rules"].show()
results["rules"][["rule", "support", "confidence", "lift"]].head(15)

,rule,support,confidence,lift
0,"Type: Professional certifications → Apprenticeship: yes + Sector: Private, no State ...",0.020858,0.831530,4.099323
1,"Apprenticeship: yes + Type: Professional certifications → Sector: Private, no State ...",0.020858,0.834109,3.711898
2,"Apprenticeship: yes + Remote teaching → Sector: Private, no State contract",0.022564,0.657627,2.926531
3,"Apprenticeship: yes + Region: Île-de-France → Sector: Private, no State contract",0.036638,0.609677,2.713149
4,"Sector: Private, no State contract + Type: Professional certifications → Apprentices...",0.020858,0.998145,2.288254
5,Type: Professional certifications → Apprenticeship: yes,0.025007,0.996909,2.285421
6,"Sector: Private, no State contract + Type: BTS (2-year technician degree) → Apprenti...",0.167642,0.989021,2.267338
7,Remote teaching + Type: BTS (2-year technician degree) → Apprenticeship: yes,0.025046,0.968516,2.220329
8,"Region: Île-de-France + Sector: Private, no State contract → Apprenticeship: yes + T...",0.031210,0.759434,2.142647
9,"Sector: Private, no State contract → Apprenticeship: yes + Type: BTS (2-year technic...",0.167642,0.746032,2.104834


**Sanity check.** Before filtering, the rules with the highest lift (about 27) say that **public CPGE ↔ boarding school**. This is a well-known fact (preparatory classes are hosted in large public high schools with boarding facilities), which shows that the algorithm finds real structure. We then keep only the rules that involve `Apprenticeship: yes` (`rules_about` function), because that is our question.

**Reading.**
- **{BTS, private without State contract} → apprenticeship**: support 0.168 (4,324 programmes), confidence 0.989, lift 2.27. In 2026, a private BTS outside the State-contract system is almost always an apprenticeship programme. These are mostly training centres that appeared after 2018.
- **{Private without State contract} → apprenticeship** alone already reaches 90% confidence: this sector *is* apprenticeship.
- **{Remote teaching, BTS} → apprenticeship** (confidence 0.97, lift 2.22): the apprentice spends most of the time in the company, so part of the teaching is done at a distance.
- **{Professional certifications} → apprenticeship** (confidence 0.997): these certifications exist almost only as apprenticeship programmes.

**Reading the Sankey.** Most flows end in `Apprenticeship: yes`. The node in the middle (`Sector: Private, no State contract`) is the consequent of some rules and part of the antecedent of others: it is the hinge between apprenticeship and the private sector.

**How could this be used?** A lift of 2.3 means the combination is 2.3 times more frequent than chance. For a policy maker, it shows that the growth of apprenticeship in the first years of higher education rests mostly on one model: the private BTS training centre.

### 8.4 Query 3: temporal analysis and forecasting

In [24]:
results["series"]["yearly"]

,annee,programmes,apprenticeship,app_share_%,app_growth_%
0,2020,16990,4263,25.1,NaN
1,2021,19439,6054,31.1,42.0
2,2022,21334,7524,35.3,24.3
3,2023,23127,9188,39.7,22.1
4,2024,24987,10812,43.3,17.7
5,2025,25827,11456,44.4,6.0
6,2026,25793,11251,43.6,-1.8


In [25]:
results["figs"]["q3_river"].show()
results["figs"]["q3_adopters"].show()
results["series"]["adopters"]

,annee,new_adopters
0,2020,1849
1,2021,622
2,2022,623
3,2023,672
4,2024,564
5,2025,389
6,2026,285


**Reading.** Apprenticeship programmes went from **4,263 in 2020 to 11,456 in 2025 (x2.7)**, and their share of the whole Parcoursup offer from 25% to 44%. The growth slowed down every year (+42%, +24%, +22%, +18%, +6%) and turned **slightly negative in 2026 (-1.8%)**. The Themeriver shows that BTS carries most of the growth. The number of new adopters also falls: about 620 to 670 institutions opened their first apprenticeship programme each year from 2021 to 2023, 389 in 2025 and 285 in 2026. This matches two external facts: the State hiring aid per apprentice was cut in February 2025 (from 6,000 € to 5,000 € in companies under 250 employees, 2,000 € above), and the Ministry of Education counted 2.8% fewer apprentices at the end of 2025 than a year before.

In [26]:
results["figs"]["q3_forecast"].show()
print("Back-test (we hide the last 1 or 2 years, predict them, and compare with reality):")
display(results["backtest"])
print("Forecast of the number of apprenticeship programmes:")
for name, fc in results["forecasts"].items():
    print(name); display(fc.assign(year=fc["ds"].dt.year)[["year", "yhat", "yhat_lower", "yhat_upper"]].tail(3).round(0))
print("Forecast of the apprenticeship share (%), flexible trend:")
results["share_forecast"].assign(year=results["share_forecast"]["ds"].dt.year)[["year", "yhat", "yhat_lower", "yhat_upper"]].tail(3).round(1)

Back-test (we hide the last 1 or 2 years, predict them, and compare with reality):


,trained_on,predicted_years,changepoint_prior_scale,predicted,real,MAPE_%
0,2020-2025,[2026],0.05,[13411],[11251],19.2
1,2020-2025,[2026],0.50,[12814],[11251],13.9
2,2020-2024,"[2025, 2026]",0.05,"[12440, 14064]","[11456, 11251]",16.8
3,2020-2024,"[2025, 2026]",0.50,"[12440, 14064]","[11456, 11251]",16.8


Forecast of the number of apprenticeship programmes:
Rigid trend (cps 0.05)


,year,yhat,yhat_lower,yhat_upper
7,2027,13656.0,12306.0,14925.0
8,2028,14907.0,13588.0,16119.0
9,2029,16161.0,14868.0,17423.0


Flexible trend (cps 0.5)


,year,yhat,yhat_lower,yhat_upper
7,2027,11657.0,11031.0,12205.0
8,2028,11905.0,10254.0,13526.0
9,2029,12154.0,9354.0,15375.0


Forecast of the apprenticeship share (%), flexible trend:


,year,yhat,yhat_lower,yhat_upper
7,2027,44.1,42.3,46.0
8,2028,44.3,39.2,50.1
9,2029,44.4,35.3,55.4


**Reading.** The back-test is the honest part.
- Trained on 2020-2024 and asked to predict 2025-2026, both settings give the same straight line (with 5 points the trend has no room to bend) and overshoot reality by 16.8% on average. **The slowdown could not be guessed from the past values alone**: it came from outside the data (the cut in hiring aids in 2025).
- Trained on 2020-2025 and asked to predict 2026, the flexible setting is closer (error 13.9% against 19.2%), because it has already seen the 2025 slowdown.

Trained on all 7 years, the two settings disagree:
- the rigid trend (default) draws a straight line through the whole period and keeps growing (13,656 programmes predicted for 2027, 16,161 for 2029);
- the flexible trend detects the change of slope after 2024 and predicts a **plateau**: 11,657 in 2027 and 12,154 in 2029, with an interval that widens with time (8,990 to 15,235 in 2029).

We consider the flexible model more credible because it reproduces the last three years. The apprenticeship share would stay around 44% of the offer. In short: the expansion phase is over, the offer is entering a **mature phase**. In a mature market, institutions grow less by opening new programmes and more by keeping their students until the diploma, which leads to our second dataset.

In [27]:
results["figs"]["q3_animation"].show()

**Reading of the animation (static state replacement).** Press "play": each frame is one Parcoursup session. Points appear everywhere between 2020 and 2023, in big cities first and then in medium-sized towns; after 2024 the map hardly changes.

### 8.5 Query 4: spatial clustering

In [28]:
results["figs"]["q4_france"].show()
results["hubs"].head(12)

,hub,sites,programmes,main_town,centre_lat,centre_lon,radius_km
0,11,467,1459,Paris,48.8426,2.3320,11.9
1,0,123,377,Lyon,45.7480,4.8516,4.0
2,9,106,316,Marseille,43.3819,5.3979,11.5
3,20,68,246,Nice,43.6565,7.1568,11.0
4,17,74,245,Lille,50.6481,3.0806,5.0
5,33,75,235,Bordeaux,44.8481,-0.5769,3.9
6,32,73,223,Toulouse,43.5949,1.4373,4.1
7,7,61,212,Montpellier,43.6089,3.8878,3.6
8,37,61,191,Nantes,47.2247,-1.5496,4.2
9,23,44,171,Rennes,48.1003,-1.6895,4.7


**Reading (national OPTICS, eps = 10 km, min_samples = 15).** OPTICS finds 49 hubs among 3,743 sites, and they are the big metropolitan areas. The Paris hub is by far the largest (467 sites, 1,459 apprenticeship programmes, about 4 times Lyon with 377); Marseille, Nice, Lille, Bordeaux, Toulouse, Montpellier and Nantes follow. 43.9% of the sites are labelled *noise*: these are isolated sites in medium and small towns, which is normal for a density method. A noise point is not an error: it means "not part of a dense urban hub".

In [29]:
results["figs"]["q4_idf_optics"].show()
results["figs"]["q4_reach"].show()
results["idf_optics"]

,optics,sites,programmes,main_town,centre_lat,centre_lon,radius_km
0,2,28,88,Paris,48.8417,2.2863,1.2
1,7,18,88,Courbevoie,48.8928,2.2490,1.0
2,12,17,73,Évry-Courcouronnes,48.6260,2.4359,2.8
3,6,19,68,Paris,48.8804,2.3038,1.1
4,13,13,53,Cergy,49.0460,2.0781,2.7
5,9,14,51,Massy,48.7283,2.2707,1.9
6,10,12,50,Noisiel,48.8422,2.6044,2.2
7,4,11,48,Paris,48.8567,2.3766,0.4
8,8,19,44,Saint-Maur-des-Fossés,48.7934,2.4523,2.7
9,0,11,39,Paris,48.8287,2.3550,0.8


In [30]:
results["figs"]["q4_silhouette"].show()
results["figs"]["q4_idf_kmeans"].show()
results["idf_kmeans"]

,kmeans,sites,programmes,main_town,centre_lat,centre_lon,radius_km
0,1,350,1067,Paris,48.8604,2.3365,7.2
1,0,68,226,Versailles,48.8990,1.9910,15.9
2,2,52,152,Évry-Courcouronnes,48.5985,2.5213,13.6
3,3,33,105,Tremblay-en-France,48.8563,2.7646,15.2


**Reading (Paris region, OPTICS vs K-means, lab exercise).**
- **OPTICS** (xi method, min_samples = 8) finds 15 small, very dense pockets with a radius of 0.4 to 3.6 km: several pockets inside Paris, the La Défense business district (Courbevoie, Puteaux), and the centres of the "new towns" built around Paris (Évry, Cergy, Marne-la-Vallée around Noisiel, Saint-Quentin-en-Yvelines around Montigny-le-Bretonneux, Massy). 56.7% of the sites are noise, because density in the outer suburbs is much lower. The reachability plot shows it: deep valleys = dense pockets, high bars = gaps.
- **K-means** with the best silhouette score gives **k = 4** territories: Paris and the inner suburbs (350 sites), the west (Versailles side, 68), the south (Évry side, 52) and the east (Marne-la-Vallée side, 33). Every site belongs to a territory, even isolated ones. The silhouette prefers few groups because Paris is one huge dense block.

The two methods answer two different needs. OPTICS answers *"where are the dense spots?"*, K-means answers *"how do I split the region into zones of similar size?"*. For instance, someone who must visit every training centre of the region (an inspector, a recruiter, a company looking for apprentices) would use the K-means territories to plan one zone per day, and the OPTICS pockets to visit many centres in a few hours on foot.

### 8.6 Extra point: second dataset (InserJeunes)

In [31]:
print("InserJeunes raw table:", results["ij_raw_shape"])
results["figs"]["ext_hist"].show()
results["enriched"]["taux_contrats_interrompus"].describe().round(1)

InserJeunes raw table: (11314, 15)


count    347.0
mean      32.5
std        9.6
min        0.0
25%       26.0
50%       32.0
75%       40.0
max       65.0
Name: taux_contrats_interrompus, dtype: float64

In [32]:
results["figs"]["ext_hubs"].show()
h = results["hubs_outcomes"]
h[h["centres_with_interruption_data"] >= 5]     # hubs where the average rests on at least 5 centres

,hub,sites,programmes,main_town,centre_lat,centre_lon,radius_km,interruption_rate_mean,centres_with_interruption_data,employment_6m_mean
0,11,467,1459,Paris,48.8426,2.3320,11.9,33.2,45,58.4
1,0,123,377,Lyon,45.7480,4.8516,4.0,24.3,9,65.9
2,9,106,316,Marseille,43.3819,5.3979,11.5,42.8,5,59.9
5,33,75,235,Bordeaux,44.8481,-0.5769,3.9,30.6,10,66.8
6,32,73,223,Toulouse,43.5949,1.4373,4.1,33.3,9,64.9
7,7,61,212,Montpellier,43.6089,3.8878,3.6,27.2,6,56.1
8,37,61,191,Nantes,47.2247,-1.5496,4.2,25.6,11,67.7
9,23,44,171,Rennes,48.1003,-1.6895,4.7,31.3,6,69.1
15,22,35,123,Dijon,47.3223,5.0427,2.8,27.0,7,66.7
17,12,39,119,Rouen,49.4374,1.0881,3.5,27.6,5,70.1


**Reading.** 956 of our 3,743 sites of 2026 are found in InserJeunes, and 347 of them have an interruption rate. For these 347 centres the **median share of apprenticeship contracts interrupted before the end is 32%** (half of the centres are between 26% and 40%, the extremes are 0% and 65%): roughly one apprentice out of three stops the contract early (change of company, dropout, failure, end of the trial period). Among the 13 hubs with at least 5 measured centres, the average goes from 24% (Lyon) to 43% (Marseille), with Paris at 33%; the employment rate 6 months after leaving goes from 56% (Montpellier) to 70% (Rouen). Paris combines a high interruption rate with one of the lowest employment rates (58%).

**Limits.** The interruption rate is only published for the 2018-2019 cohorts, and only for part of the centres (many centres opened after 2019 and do not have it). It is an interpretation aid, not a precise measure for 2026.

**Link with the previous queries.** Query 3 showed that the offer stopped growing; this dataset shows that a large share of contracts end early. Together, they suggest that the next challenge for the apprenticeship system is **retention and follow-up of apprentices during their training** rather than the opening of new programmes.

## 9. Conclusion

| Query | Technique | Main result |
|---|---|---|
| Q1 Grouping | groupby, counts, shares | 11,251 apprenticeship programmes in 2026 (44% of the Parcoursup offer); BTS = 81% of them; Auvergne-Rhône-Alpes and Île-de-France tied first; private training centres lead |
| Q2 Pattern mining | Apriori, association rules, Sankey | {BTS, private without State contract} → apprenticeship (support 0.17, confidence 0.99, lift 2.27) |
| Q3 Temporal | Themeriver, static state replacement, Prophet | x2.7 from 2020 to 2025, then -1.8% in 2026; the flexible Prophet trend forecasts a plateau (11,657 in 2027) |
| Q4 Spatial | OPTICS (haversine), K-means (silhouette) | 49 national hubs = metropolitan areas, Paris 4 times Lyon; 15 dense pockets and 4 territories in the Paris region |
| Extra | InserJeunes, join on UAI | median 32% of contracts interrupted before the end (347 centres) |

**Take-away.** Apprenticeship has become a mainstream way to enter higher education in France, but its expansion is over. The challenge now moves from *opening programmes* to *keeping apprentices until the diploma*.

**What we would do next.** Use monthly data on new apprenticeship contracts (DARES) to forecast seasonality (most contracts start in September and October), and follow the interruption rate when it is published for recent cohorts.

**Sources.** Ministère de l'Enseignement supérieur et de la Recherche, *Cartographie des formations Parcoursup* (data.enseignementsup-recherche.gouv.fr); Ministère de l'Éducation nationale (DEPP) and DARES, *InserJeunes Apprentissage par CFA* (data.education.gouv.fr); Fayyad, Piatetsky-Shapiro, Smyth (1996), *From Data Mining to Knowledge Discovery in Databases*, AI Magazine.